In [61]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error # MAE

# Загрузка данных

In [2]:
# %%capture
# !wget https://www.dropbox.com/s/64ol9q9ssggz6f1/data_ford_price.xlsx

In [4]:
ford_price_data = pd.read_excel('Data/data_ford_price.xlsx')

# Предобработка: удаление пропусков

Для начала проверим наш изначальный датасет на пропуски

In [5]:
display(ford_price_data.isnull().sum())

price              0
year               0
condition          0
cylinders          0
odometer           0
title_status       0
transmission       0
drive            391
size            1564
lat                0
long               0
weather          180
dtype: int64

Для начала просто попробуем убрать все пропущенные значения, это делается намеренно.

In [6]:
# ~ - тильда означает, что мы выбираем все строки датасета
df = ford_price_data[~ford_price_data['weather'].isna()]
display(df.shape)

(6837, 12)

Число строк сократилось до 6837, что безусловно плохо, т.к. мы простым удалением уменьшили датасет, часть информации просто потеряна!

# Очистка данных

Наибольшая сложность в очистке данных от пропусков заключается в выборе метода их обработки.

<img src='Images/ml_03.png'>

Первым делом воспользуемся методом удаления строк с пропусками. Плюс данного метода состоит в том, что модель, обученная с удалением всех пропущенных значений, является надёжной, то есть имеет сравнительно хорошее качество на тесте. Среди минусов — потеря большого количества информации, а также плохое качество работы, если процент отсутствующих значений слишком велик по сравнению с полным набором данных.

In [8]:
cols = ['size', 'weather', 'drive']
for col in cols: 
    missing_percent = ford_price_data[col].isna().mean() * 100
    print(f'{col}: {missing_percent:.2f} %')

size: 22.29 %
weather: 2.57 %
drive: 5.57 %


В качестве регрессора воспользуемся линейной моделью, а качество оценим с помощью коэффициента детерминации. Также нам потребуется разделить модель на обучающую и тестовую выборки.

In [9]:
X = ford_price_data.drop(columns='price')
y = ford_price_data['price']

Удалим данные с пропусками:

In [10]:
X = X.dropna()

Тоже самое делаем для целевой переменной

In [11]:
y = y.iloc[X.index]

In [14]:
# Проверим одинаковую длинну в выборках
display(X.shape[0] == y.shape[0])

True

Разделим выборку на обучающую и тестовую.

In [15]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=30)

# Кодирование категориальных переменных

In [19]:
columns_to_change = ['cylinders', 'title_status', 'transmission', 'drive', 'size']

In [21]:
def encode_cat_features(columns_to_change, X_train, X_test, y_train):
    # Проведём кодирование OneHot-методом категориальных переменных.
    one_hot_encoder = OneHotEncoder()
    # Обучаем энкодер и сразу применяем преобразование к выборке. Результаты переводим в массивы
    X_train_onehot = one_hot_encoder.fit_transform(X_train[columns_to_change]).toarray()
    X_test_onehot = one_hot_encoder.transform(X_test[columns_to_change]).toarray()
    
    # Для удобства сохраним полученные названия новых колонок в отдельную переменную
    columns = one_hot_encoder.get_feature_names_out(columns_to_change)
    
    # Теперь у нас есть массив закодированных признаков и наша изначальная таблица. 
    # Чтобы соединить эти данные, переведём массив в формат DataFrame.
    X_train_onehot_df = pd.DataFrame(X_train_onehot, columns=columns)
    X_test_onehot_df = pd.DataFrame(X_test_onehot, columns=columns)

    X_train = X_train.reset_index().drop(['index'], axis = 1)
    X_test = X_test.reset_index().drop(['index'], axis = 1)
    y_train = y_train.reset_index().drop(['index'], axis = 1)

    # Объединяем таблицы и удаляем старые категориальные признаки
    X_train_new = pd.concat([X_train, X_train_onehot_df], axis=1)
    X_test_new = pd.concat([X_test, X_test_onehot_df], axis=1)
    
    X_train_new = X_train_new.drop(columns=columns_to_change)
    X_test_new = X_test_new.drop(columns=columns_to_change)

    return X_train_new, X_test_new

In [23]:
X_train_new, X_test_new = encode_cat_features(columns_to_change, X_train, X_test, y_train)

# Обучение модели

Настало время обучить модель. Для этого создаём объект класса LinearRegression.

In [24]:
lr_model = LinearRegression()

In [25]:
# Обучаем модель по МНК (метод наименьших квадратов)
lr_model.fit(X_train_new, y_train)
# Делаем предсказание
y_train_predict = lr_model.predict(X_train_new)
y_test_predict = lr_model.predict(X_test_new)
# Выводим метрики R2
print("Train R^2: {:.3f}".format(r2_score(y_train, y_train_predict)))
print("Test R^2: {:.3f}".format(r2_score(y_test, y_test_predict)))

Train R^2: 0.647
Test R^2: 0.693


# Предобработка: заполнение пропусков

Теперь давайте попробуем заполнить пропуски константными значениями и обучить модель заново. Плюс такого подхода состоит в том, что мы предотвращаем потерю данных, которая происходит при удалении строк или столбцов. Основной минус — в снижении разброса (разнообразия) признаков.

In [26]:
X = ford_price_data.drop(columns='price')
y = ford_price_data['price']

In [27]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=30)

In [28]:
X_train['weather'] = X_train['weather'].fillna(np.round(np.mean(X_train['weather']), 0))
X_test['weather'] = X_test['weather'].fillna(np.round(np.mean(X_train['weather']), 0))

Для простоты воспользуемся заполнением наиболее частым значением категориальных признаков. Для этого сначала определим их в наших признаках, использовав комбинацию методов value_counts() и head()

In [31]:
X_train['drive'].value_counts(True).head(1)

drive
4wd    0.738406
Name: proportion, dtype: float64

In [32]:
X_train['size'].value_counts(True).head(1)

size
full-size    0.841164
Name: proportion, dtype: float64

In [33]:
X_train['size'] = X_train['size'].fillna('full-size')
X_train['drive'] = X_train['drive'].fillna('4wd')

X_test['size'] = X_test['size'].fillna('full-size')
X_test['drive'] = X_test['drive'].fillna('4wd')

In [34]:
X_train_new, X_test_new = encode_cat_features(columns_to_change, X_train, X_test, y_train)

In [35]:
lr_model = LinearRegression()
lr_model.fit(X_train_new, y_train)
y_train_predict = lr_model.predict(X_train_new)
y_test_predict = lr_model.predict(X_test_new)
print("Train R^2: {:.3f}".format(r2_score(y_train, y_train_predict)))
print("Test R^2: {:.3f}".format(r2_score(y_test, y_test_predict)))

Train R^2: 0.649
Test R^2: 0.465


Примечание: модели с коэффициентом детерминации выше 0.8 можно признать достаточно хорошими. Равенство коэффициента детерминации 1 означает, что объясняемая переменная в точности описывается рассматриваемой моделью.

# Предсказание пропусков с помощью ML

Приведённые методы обработки отсутствующих значений не учитывают корреляционную связь признака, содержащего пропуски, с остальными. Признаки, не имеющие NaN, можно использовать для прогнозирования пропущенных значений. Строится модель регрессии или классификации в зависимости от характера (категорийного или непрерывного) признака, имеющего пропущенное значение.

In [ ]:
# Скопируем данные в отдельную переменную
data = X.copy()

# В качестве тестовой выборки возьмем строки с пропусками в признаке weather
test_data = data[data['weather'].isnull()]
# И удалим эти строчки из таблицы
data.dropna(inplace=True)

# Определим целевой признак и факторы
y_train = data['weather']
X_train = data.drop(['size', 'weather', 'drive'], axis=1)
X_test = test_data.drop(['size', 'weather', 'drive'], axis=1)

In [37]:
# Создадим кодировщик
one_hot_encoder = OneHotEncoder()
categorial_cols = ['cylinders', 'title_status', 'transmission']

In [38]:
# Закодируем категориальные признаки
X_train_new, X_test_new = encode_cat_features(categorial_cols, X_train, X_test, y_train)

In [39]:
# Создадим модель линейной регрессии и обучим ее на задачу предсказания пропусков
model = LinearRegression()
model.fit(X_train_new, y_train)

# Сделаем предсказание целевой переменной (пропущенных значений в признаке weather)
y_pred = model.predict(X_test_new)
y_pred

array([ 40.91435555,  40.7637233 ,  39.74866152,  41.2755305 ,
        40.31791932,  41.10796547,  41.15337846,  39.94866488,
        41.10796547,  40.7217165 ,  40.18904454,  91.62094167,
        41.12549856,  41.33052316,  39.66827354,  40.91435555,
        40.77287826,  40.84208674,  41.10796547,  41.02118034,
        40.31791932,  41.30309209,  40.77645269,  40.75842615,
        40.61605044,  40.79031628,  40.7701239 ,  39.78723017,
        41.27231621,  39.77492057,  40.7637233 ,  40.7701239 ,
        41.10796547,  39.68313064,  40.12277414,  39.7873657 ,
        41.07798631,  41.06812063,  40.7637233 ,  40.90194049,
        41.10796547,  70.85737739,  30.44339508,  40.76125291,
        39.77492057,  41.12549856,  39.74864816,  40.72219488,
        40.97162064,  40.7448585 ,  39.71065847,  39.78779447,
        39.77492057,  40.15945849,  41.12549856,  40.7902779 ,
        39.65555168,  41.10796547,  40.76125291,  40.15619215,
        40.79031628,  41.10796547,  40.7902779 ,  40.72

# Дополнительно

Вставим найденную замену на место пропусков в столбце weather. Используем тот же метод для заполнения пропусков в size. Стоит обратить внимание на тип модели, который нужен (классификация или регрессия) в зависимости от типа признака.

In [52]:
for i, col in enumerate(test_data.index): # Возьмем индексы с датафрейна с целевой переменой weather
    X.loc[col, 'weather'] = y_pred[i]

In [53]:
X.isnull().sum() # Проверям, что weather теперь без пропусков

year               0
condition          0
cylinders          0
odometer           0
title_status       0
transmission       0
drive            391
size            1564
lat                0
long               0
weather            0
dtype: int64

Тоже самое теперь для size. Данный признак является категориальным. Следовательно, понадобится классификатор для заполения пропусков в нем.

In [55]:
# Скопируем данные в отдельную переменную
data = X.copy()

# В качестве тестовой выборки возьмем строки с пропусками в признаке size (категориальный признак)
test_data = data[data['size'].isnull()]
# И удалим эти строчки из таблицы
data.dropna(inplace=True)

# Определим целевой признак и факторы
y_train = data['size']
X_train = data.drop(['size', 'drive'], axis=1)
X_test = test_data.drop(['size', 'drive'], axis=1)

# Создадим кодировщик
one_hot_encoder = OneHotEncoder()
categorial_cols = ['cylinders', 'title_status', 'transmission']

# Закодируем категориальные признаки
X_train_new, X_test_new = encode_cat_features(categorial_cols, X_train, X_test, y_train)

# Создадим модель логистической регрессии и обучим ее на задачу предсказания пропусков
model = LogisticRegression(max_iter=1_000)
model.fit(X_train_new, y_train)

# Сделаем предсказание целевой переменной (пропущенных значений в признаке size)
y_pred = model.predict(X_test_new)
y_pred

/Users/alexander/.pyenv/versions/3.10.14/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:470: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


array(['full-size', 'full-size', 'full-size', ..., 'full-size',
       'full-size', 'full-size'], dtype=object)

In [56]:
for i, col in enumerate(test_data.index): # Возьмем индексы с датафрейна с целевой переменой size
    X.loc[col, 'size'] = y_pred[i]

In [57]:
X.isnull().sum()

year              0
condition         0
cylinders         0
odometer          0
title_status      0
transmission      0
drive           391
size              0
lat               0
long              0
weather           0
dtype: int64

Теперь признак size, тоже без пропусков

# Пребодработка: работа с выбросами

Помимо пропусков, на пути анализа данных всплывает ещё один подводный камень — выбросы (аномалии).

In [58]:
data = pd.read_excel('Data/data_ford_price.xlsx') 

In [59]:
data.describe()

,price,year,condition,cylinders,odometer,lat,long,weather
count,7017.000000,7017.000000,7017.000000,7017.000000,7.017000e+03,7017.000000,7017.000000,6837.000000
mean,15121.549523,2007.869745,2.598689,7.374662,1.199787e+05,39.550144,-91.903404,52.142899
std,11765.423119,6.975329,0.703662,0.942928,8.992216e+04,5.745409,14.030710,7.954830
min,1.000000,1957.000000,0.000000,3.000000,0.000000e+00,-2.508807,-151.055832,29.000000
25%,5995.000000,2004.000000,2.000000,6.000000,7.328500e+04,35.661076,-95.937145,45.000000
50%,12750.000000,2010.000000,3.000000,8.000000,1.180000e+05,40.335245,-88.168416,51.000000
75%,21995.000000,2013.000000,3.000000,8.000000,1.578040e+05,43.582100,-82.706300,59.000000
max,299500.000000,2018.000000,5.000000,10.000000,2.490000e+06,77.617682,-5.377999,71.000000


Выбросы могут искажать статистические показатели и распределения данных. Удаление выбросов из обучающих данных перед моделированием может привести к росту качества прогнозов.

К счастью, существуют автоматические, основанные на моделях методы выявления выбросов, которые уже имплементированы в sklearn.

Для начала сформируем baseline-модель. Проведём следующую предобработку: для простоты уберём категориальные столбцы из данных и затем удалим строки с пропусками.

In [65]:
data = data[['price', 'year', 'cylinders', 'odometer' ,'lat', 'long','weather']]
data.dropna(inplace = True)

In [66]:
X = data.drop(columns='price')
y = data['price']

X.head()

,year,cylinders,odometer,lat,long,weather
0,2016,6,43500,36.471500,-82.483400,59.0
1,2009,8,98131,40.468826,-74.281734,52.0
2,2002,8,201803,42.477134,-82.949564,45.0
3,2000,8,170305,40.764373,-82.349503,49.0
5,2003,8,167662,45.518031,-122.578752,50.0


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=40)

MAE: 4682.957


Настало время обработки выбросов.

Первый алгоритм, который мы применим, — __Isolation Forest__, или iForest. Это алгоритм обнаружения аномалий на основе дерева.

Данный метод стремится изолировать аномалии, которые немногочисленны и различаются по пространству признаков.

Одним из основных гиперпараметров модели является __contamination («загрязнение»)__, который используется для оценки количества выбросов в наборе данных. Его значение находится в диапазоне от 0.0 до 0.5 и по умолчанию равно 0.1.

In [ ]:
from  sklearn.ensemble import IsolationForest

# ищем выбросы в обучающей выборке
iso = IsolationForest(contamination=0.1)
iso.fit(X_train.values)
y_predicted = iso.predict(X_train.values)

In [ ]:
# выберем все строки, которые не являются выбросами
mask = y_predicted != -1
X_train, y_train = X_train[mask], y_train[mask]

print(X_train.shape, y_train.shape)
 
model = LinearRegression()
model.fit(X_train, y_train)
 
y_predicted = model.predict(X_test)
mae = mean_absolute_error(y_test, y_predicted)
print('MAE: %.3f' % mae)

(4308, 6) (4308,)
MAE: 4425.198


Следующий метод — __Local Outlier Factor__, или LOF. Это метод, который пытается использовать идею ближайших соседей для обнаружения выбросов.

Каждому примеру присваивается оценка того, насколько он изолирован от его локальных соседей. Примеры, которые наиболее отдалены от соседей, скорее всего, будут являться выбросами.

Библиотека scikit-learn обеспечивает реализацию этого подхода в классе LocalOutlierFactor.

In [70]:
from sklearn.neighbors import LocalOutlierFactor
 
lof = LocalOutlierFactor()
y_predicted = lof.fit_predict(X_train)

# Также отбираем данные, которые не являются выбросами (-1)
mask = y_predicted != -1
X_train, y_train = X_train[mask], y_train[mask]

print(X_train.shape, y_train.shape)
 
model = LinearRegression()
model.fit(X_train, y_train)
 
y_predicted = model.predict(X_test)
mae = mean_absolute_error(y_test, y_predicted)
print('MAE: %.3f' % mae)

(3966, 6) (3966,)
MAE: 4423.187


/Users/alexander/.pyenv/versions/3.10.14/lib/python3.10/site-packages/sklearn/neighbors/_lof.py:322: UserWarning: Duplicate values are leading to incorrect results. Increase the number of neighbors for more accurate results.
  warnings.warn(


Напоследок рассмотрим __Minimum Covariance Determinant__, или MCD.

Эффективная реализация этого метода для многомерных данных известна как детерминант минимальной ковариации (MCD).

Библиотека scikit-learn предоставляет доступ к этому методу через класс EllipticEnvelope.

In [71]:
from sklearn.covariance import EllipticEnvelope

ee = EllipticEnvelope(contamination=0.01)
y_predicted = ee.fit_predict(X_train)

# Без выбросов
mask = y_predicted != -1
X_train, y_train = X_train[mask], y_train[mask]

print(X_train.shape, y_train.shape)

model = LinearRegression()
model.fit(X_train, y_train)
 
y_predicted = model.predict(X_test)
mae = mean_absolute_error(y_test, y_predicted)
print('MAE: %.3f' % mae)

(3926, 6) (3926,)
MAE: 4436.652


Данные алгоритмы носят стохастический характер, поэтому результаты метрики могут отличаться от прогона к прогону!

Мы видим, что оптимальный результат достигается с помощью древовидного алгоритма Isolation Forest, тогда как пространственные методы LOF и MCD принимают за выбросы больше данных, что приводит к ухудшению качества. Тем не менее все три метода превосходят baseline.

<img src='Images/ml_04.png'>